## Link scATAC-seq peaks to genes using correlation

- last updated: 1/15/2025
- Reference: https://stuartlab.org/signac/reference/linkpeaks
- Reference: https://stuartlab.org/signac/articles/pbmc_multiomic#linking-peaks-to-genes 

### Description: 
Find peaks that are correlated with the expression of nearby genes. For each gene, this function computes the correlation coefficient between the gene expression and accessibility of each peak within a given distance from the gene TSS, and computes an expected correlation coefficient for each peak given the GC content, accessibility, and length of the peak. The expected coefficient values for the peak are then used to compute a z-score and p-value.

In [22]:
suppressMessages(library(Signac))
suppressMessages(library(Seurat))
suppressMessages(library(GenomeInfoDb))
library(Rsamtools)  # for FaFile()

library(ggplot2)
library(dplyr)
library(RColorBrewer)
library(patchwork)
library(stringr)
library(VennDiagram)
library(GenomicRanges)

# zebrafish genome
library(BSgenome.Drerio.UCSC.danRer11)

In [93]:
seurat <- readRDS("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/01_Signac_processed/integrated_RNA_ATAC_wnn_gene_activity_3d_umaps.rds")
seurat

In [94]:
seurat

An object of class Seurat 
726063 features across 95196 samples within 5 assays 
Active assay: peaks_integrated (640834 features, 0 variable features)
 4 other assays present: RNA, SCT, integrated, Gene.Activity
 9 dimensional reductions calculated: integrated_lsi, umap, integrated_pca, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

In [96]:
# List of assays to keep
assaysToKeep <- c("peaks_integrated", "RNA", "SCT")

# Function to remove other assays from a Seurat object
remove_unwanted_assays <- function(seuratObject, assaysToKeep) {
  allAssays <- names(seuratObject@assays)
  assaysToRemove <- setdiff(allAssays, assaysToKeep)
  
  for (assay in assaysToRemove) {
    seuratObject[[assay]] <- NULL
  }
  
  return(seuratObject)
}

# Apply the function to each of your Seurat objects
seurat <- remove_unwanted_assays(seurat, assaysToKeep)
seurat

An object of class Seurat 
699031 features across 95196 samples within 3 assays 
Active assay: peaks_integrated (640834 features, 0 variable features)
 2 other assays present: RNA, SCT
 8 dimensional reductions calculated: integrated_lsi, umap, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

In [95]:
# path to your custom genome
custom_fa_path <- "/hpc/reference/sequencing_alignment/alignment_references/zebrafish_genome_GRCz11/fasta/genome.fa"
GRCz11 <- FaFile(custom_fa_path)

In [100]:
# Filepath to your CSV
file_path <- "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/05_SEACells_processed/objects_75cells_per_metacell/TDR118reseq_seacells_obs_annotation_ML_coarse.csv"

# Load the CSV into a data frame
df <- read.csv(file_path)

# Check the column names to identify the relevant ones
colnames(df)

# Extract the columns for cell-to-metacell mapping (adjust column names based on the actual file structure)
cell_metacell_mapping <- df[, c("index", "SEACell")] 
cell_metacell_mapping %>% head()

[1] "index"                   "orig.ident"             
 [3] "nCount_RNA"              "nFeature_RNA"           
 [5] "nCount_ATAC"             "nFeature_ATAC"          
 [7] "nucleosome_signal"       "nucleosome_percentile"  
 [9] "TSS.enrichment"          "TSS.percentile"         
[11] "nCount_SCT"              "nFeature_SCT"           
[13] "global_annotation"       "nCount_peaks_bulk"      
[15] "nFeature_peaks_bulk"     "nCount_peaks_celltype"  
[17] "nFeature_peaks_celltype" "nCount_peaks_merged"    
[19] "nFeature_peaks_merged"   "SCT.weight"             
[21] "peaks_merged.weight"     "nCount_Gene.Activity"   
[23] "nFeature_Gene.Activity"  "annotation_ML"          
[25] "scANVI_zscape"           "annotation_ML_coarse"   
[27] "dev_stage"               "SEACell"

,index,SEACell
,<chr>,<chr>
1,AAACAGCCACCTAAGC-1,SEACell-85
2,AAACAGCCAGGGAGGA-1,SEACell-45
3,AAACAGCCATAGACCC-1,SEACell-178
4,AAACATGCAAACTCAT-1,SEACell-34
5,AAACATGCAAGGACCA-1,SEACell-128
6,AAACATGCAAGGATTA-1,SEACell-2


In [103]:
seurat@meta.data %>% head()

,orig.ident,nCount_RNA,nFeature_RNA,nCount_ATAC,nFeature_ATAC,nucleosome_signal,nucleosome_percentile,TSS.enrichment,TSS.percentile,nCount_SCT,⋯,nCount_peaks_integrated,nFeature_peaks_integrated,dataset,prediction.score.Germline,integrated.weight,peaks_integrated.weight,wsnn_res.0.8,seurat_clusters,annotation_ML,annotation_ML_coarse
,<chr>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<chr>,<chr>
AAACAGCCACCTAAGC-1_1,SeuratProject,6522,2317,21425,9781,0.5719844,0.40,4.488213,0.48,5661,⋯,13987,11974,TDR118,NA,4.062589e-01,0.5937411,10,10,epidermis,epidermis
AAACAGCCAGGGAGGA-1_1,SeuratProject,6100,2319,10334,5028,0.4481434,0.13,4.795205,0.75,5553,⋯,6889,6302,TDR118,NA,7.616462e-01,0.2383538,16,16,pronephros,pronephros
AAACAGCCATAGACCC-1_1,SeuratProject,12581,3467,51485,19874,0.5142133,0.24,5.238692,0.92,5781,⋯,32040,23386,TDR118,NA,5.938950e-08,0.9999999,14,14,hindbrain,hindbrain
AAACATGCAAACTCAT-1_1,SeuratProject,5642,2145,19812,9183,0.6733186,0.85,4.409525,0.41,5363,⋯,13090,11254,TDR118,NA,3.694048e-01,0.6305952,0,0,axial_mesoderm,axial_mesoderm
AAACATGCAAGGACCA-1_1,SeuratProject,2691,838,5182,2565,0.3949045,0.06,4.939061,0.83,4727,⋯,3390,3149,TDR118,NA,3.265597e-02,0.9673440,3,3,neural_optic2,neural_optic
AAACATGCAAGGATTA-1_1,SeuratProject,4233,1703,24072,10949,0.6424510,0.72,4.636479,0.62,4729,⋯,15383,13086,TDR118,NA,5.023156e-01,0.4976844,4,4,neural_floor_plate,neural_floor_plate


In [104]:
seurat$dataset %>% unique()

[1] "TDR118" "TDR119" "TDR124" "TDR125" "TDR126" "TDR127" "TDR128"

In [105]:
# Full list of data IDs
data_ids <- c('TDR118', 'TDR119', 'TDR124', 'TDR125', 'TDR126', 'TDR127', 'TDR128')

# Base directory for the CSV files
base_dir <- "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/05_SEACells_processed/objects_75cells_per_metacell/"

# Initialize an empty list to store data frames
dataframes <- list()

# Loop through each data ID
for (data_id in data_ids) {
  # Construct possible file paths
  if (data_id %in% c('TDR118', 'TDR119', 'TDR124', 'TDR125')) {
    # For datasets with potential "reseq"
    file_path_reseq <- paste0(base_dir, data_id, "reseq_seacells_obs_annotation_ML_coarse.csv")
    file_path_normal <- paste0(base_dir, data_id, "_seacells_obs_annotation_ML_coarse.csv")
  } else {
    # For datasets without "reseq"
    file_path_reseq <- NULL
    file_path_normal <- paste0(base_dir, data_id, "_seacells_obs_annotation_ML_coarse.csv")
  }
  
  # Determine the correct file path to use
  if (!is.null(file_path_reseq) && file.exists(file_path_reseq)) {
    file_path <- file_path_reseq
  } else if (file.exists(file_path_normal)) {
    file_path <- file_path_normal
  } else {
    warning(paste("No file found for:", data_id))
    next
  }
  
  # Load the CSV file
  df <- read.csv(file_path)
  
  # Check if the required columns exist
  if (!("index" %in% colnames(df)) || !("SEACell" %in% colnames(df))) {
    stop(paste("Missing required columns in file:", file_path))
  }
  
  # Append the dataset suffix to the 'index' column to match Seurat's naming convention
  df$index <- paste0(df$index, "_", which(data_ids == data_id))  # Appends _1, _2, etc., based on dataset order
  
  # Append the dataset suffix to the 'SEACell' column for compatibility
  df$SEACell <- paste0(df$SEACell, "_", which(data_ids == data_id))
  
  # Add the dataset ID as a separate column for tracking
  df$data_id <- data_id
  
  # Append to the list of data frames
  dataframes[[data_id]] <- df[, c("index", "SEACell", "data_id")]
}

# Concatenate all data frames into one
concatenated_df <- do.call(rbind, dataframes)

In [106]:
concatenated_df %>% head()

,index,SEACell,data_id
,<chr>,<chr>,<chr>
TDR118.1,AAACAGCCACCTAAGC-1_1,SEACell-85_1,TDR118
TDR118.2,AAACAGCCAGGGAGGA-1_1,SEACell-45_1,TDR118
TDR118.3,AAACAGCCATAGACCC-1_1,SEACell-178_1,TDR118
TDR118.4,AAACATGCAAACTCAT-1_1,SEACell-34_1,TDR118
TDR118.5,AAACATGCAAGGACCA-1_1,SEACell-128_1,TDR118
TDR118.6,AAACATGCAAGGATTA-1_1,SEACell-2_1,TDR118


In [109]:
# Save the concatenated mapping to a CSV file
write.csv(concatenated_df, "concatenated_cell_metacell_mapping.csv", row.names = FALSE)

In [112]:
# Load the concatenated mapping
mapping <- read.csv("concatenated_cell_metacell_mapping.csv")

# Check the structure of the mapping
head(mapping)

# Ensure the Seurat object has matching cell names
seurat_cells <- colnames(seurat)
mapping_cells <- mapping$index

# Find common cells between Seurat object and the mapping
common_cells <- intersect(seurat_cells, mapping_cells)

# Subset the mapping to include only cells present in the Seurat object
mapping_filtered <- mapping[mapping$index %in% common_cells, ]

# Reorder the mapping to match the Seurat object cell names
mapping_filtered <- mapping_filtered[match(seurat_cells, mapping_filtered$index), ]

# Add the SEACell column to the Seurat meta.data
seurat$SEACell <- mapping_filtered$SEACell

# Verify the updated meta.data
head(seurat@meta.data)

,index,SEACell,data_id
,<chr>,<chr>,<chr>
1,AAACAGCCACCTAAGC-1_1,SEACell-85_1,TDR118
2,AAACAGCCAGGGAGGA-1_1,SEACell-45_1,TDR118
3,AAACAGCCATAGACCC-1_1,SEACell-178_1,TDR118
4,AAACATGCAAACTCAT-1_1,SEACell-34_1,TDR118
5,AAACATGCAAGGACCA-1_1,SEACell-128_1,TDR118
6,AAACATGCAAGGATTA-1_1,SEACell-2_1,TDR118


,orig.ident,nCount_RNA,nFeature_RNA,nCount_ATAC,nFeature_ATAC,nucleosome_signal,nucleosome_percentile,TSS.enrichment,TSS.percentile,nCount_SCT,⋯,nFeature_peaks_integrated,dataset,prediction.score.Germline,integrated.weight,peaks_integrated.weight,wsnn_res.0.8,seurat_clusters,annotation_ML,annotation_ML_coarse,SEACell
,<chr>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<int>,<chr>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<chr>,<chr>,<chr>
AAACAGCCACCTAAGC-1_1,SeuratProject,6522,2317,21425,9781,0.5719844,0.40,4.488213,0.48,5661,⋯,11974,TDR118,NA,4.062589e-01,0.5937411,10,10,epidermis,epidermis,SEACell-85_1
AAACAGCCAGGGAGGA-1_1,SeuratProject,6100,2319,10334,5028,0.4481434,0.13,4.795205,0.75,5553,⋯,6302,TDR118,NA,7.616462e-01,0.2383538,16,16,pronephros,pronephros,SEACell-45_1
AAACAGCCATAGACCC-1_1,SeuratProject,12581,3467,51485,19874,0.5142133,0.24,5.238692,0.92,5781,⋯,23386,TDR118,NA,5.938950e-08,0.9999999,14,14,hindbrain,hindbrain,SEACell-178_1
AAACATGCAAACTCAT-1_1,SeuratProject,5642,2145,19812,9183,0.6733186,0.85,4.409525,0.41,5363,⋯,11254,TDR118,NA,3.694048e-01,0.6305952,0,0,axial_mesoderm,axial_mesoderm,SEACell-34_1
AAACATGCAAGGACCA-1_1,SeuratProject,2691,838,5182,2565,0.3949045,0.06,4.939061,0.83,4727,⋯,3149,TDR118,NA,3.265597e-02,0.9673440,3,3,neural_optic2,neural_optic,SEACell-128_1
AAACATGCAAGGATTA-1_1,SeuratProject,4233,1703,24072,10949,0.6424510,0.72,4.636479,0.62,4729,⋯,13086,TDR118,NA,5.023156e-01,0.4976844,4,4,neural_floor_plate,neural_floor_plate,SEACell-2_1


In [113]:
# Verify the updated meta.data
tail(seurat@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,nCount_ATAC,nFeature_ATAC,nucleosome_signal,nucleosome_percentile,TSS.enrichment,TSS.percentile,nCount_SCT,⋯,nFeature_peaks_integrated,dataset,prediction.score.Germline,integrated.weight,peaks_integrated.weight,wsnn_res.0.8,seurat_clusters,annotation_ML,annotation_ML_coarse,SEACell
,<chr>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<int>,<chr>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<chr>,<chr>,<chr>
TTTGTGTTCATGGTGT-1_7,SeuratProject,5097,1803,43974,18650,1.0779747,0.89,3.762727,0.08,3786,⋯,22804,TDR128,NA,0.09478771,0.9052123,32,32,hemangioblasts,hemangioblasts,SEACell-50_7
TTTGTGTTCCCTCAGT-1_7,SeuratProject,3079,1281,36101,16047,0.9833127,0.63,4.188447,0.44,3154,⋯,19040,TDR128,NA,0.32890016,0.6710998,1,1,tail_bud,tail_bud,SEACell-116_7
TTTGTTGGTACCTTAC-1_7,SeuratProject,6276,2441,71447,28246,0.9854497,0.63,4.178813,0.43,4014,⋯,33895,TDR128,NA,0.45076052,0.5492395,25,25,lateral_plate_mesoderm,lateral_plate_mesoderm,SEACell-58_7
TTTGTTGGTATTGAGT-1_7,SeuratProject,2377,1164,49177,19951,1.0780009,0.89,3.923300,0.17,2868,⋯,23888,TDR128,NA,0.15563581,0.8443642,5,5,neural_posterior,neural_posterior,SEACell-56_7
TTTGTTGGTGCGCGTA-1_7,SeuratProject,1750,811,12537,5794,0.6384720,0.07,4.243214,0.51,2825,⋯,6857,TDR128,NA,0.00615323,0.9938468,11,11,endocrine_pancreas,endocrine_pancreas,SEACell-62_7
TTTGTTGGTTAAGGCC-1_7,SeuratProject,8999,2399,3744,1894,0.7868852,0.17,3.665425,0.04,3638,⋯,2244,TDR128,NA,0.56467202,0.4353280,9,9,somites,somites,SEACell-126_7


In [114]:
seurat

An object of class Seurat 
699031 features across 95196 samples within 3 assays 
Active assay: peaks_integrated (640834 features, 0 variable features)
 2 other assays present: RNA, SCT
 8 dimensional reductions calculated: integrated_lsi, umap, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

In [115]:
## filter for the 50K highly variable peaks

# Load the list of filtered peaks from the file
filtered_peaks <- readLines("peaks_hvp_50k.txt")
filtered_peaks %>% head()

[1] "1-30119-30406"   "1-102102-102943" "1-128797-129190" "1-217215-217640"
[5] "1-256849-257455" "1-299178-299377"

In [117]:
seurat

An object of class Seurat 
699031 features across 95196 samples within 3 assays 
Active assay: peaks_integrated (640834 features, 0 variable features)
 2 other assays present: RNA, SCT
 8 dimensional reductions calculated: integrated_lsi, umap, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

In [119]:
# Load the progress package
library(progress)

In [118]:
# Subset the Seurat object to only include peaks in the filtered list
seurat[["peaks_integrated"]] <- subset(seurat[["peaks_integrated"]], features = filtered_peaks)
seurat

Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from peaks_integrated_ to peaksintegrated_”


An object of class Seurat 
108197 features across 95196 samples within 3 assays 
Active assay: peaks_integrated (50000 features, 0 variable features)
 2 other assays present: RNA, SCT
 8 dimensional reductions calculated: integrated_lsi, umap, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

In [ ]:
# Aggregate the RNA assay by SEACell
rna_aggregated <- AverageExpression(
  object = seurat_obj,
  assay = "RNA",
  group.by = "SEACell",
  slot = "data"  # Use normalized data; change to "counts" for raw counts if needed
)

# Aggregate the SCT assay by SEACell
sct_aggregated <- AverageExpression(
  object = seurat_obj,
  assay = "SCT",
  group.by = "SEACell",
  slot = "data"
)

# Aggregate the peaks_integrated assay by SEACell
peaks_aggregated <- AverageExpression(
  object = seurat_obj,
  assay = "peaks_integrated",
  group.by = "SEACell",
  slot = "data"
)

# Save the aggregated data to CSV files for further analysis if needed
write.csv(as.data.frame(rna_aggregated$RNA), "RNA_aggregated_by_SEACell.csv", row.names = TRUE)
write.csv(as.data.frame(sct_aggregated$SCT), "SCT_aggregated_by_SEACell.csv", row.names = TRUE)
write.csv(as.data.frame(peaks_aggregated$peaks_integrated), "peaks_integrated_aggregated_by_SEACell.csv", row.names = TRUE)

In [91]:
seurat@assays$

An object of class Seurat 
50000 features across 95196 samples within 1 assay 
Active assay: peaks_integrated (50000 features, 0 variable features)
 8 dimensional reductions calculated: integrated_lsi, umap, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

In [ ]:
# Remove out-of-bound peaks
seurat <- RemoveOutOfBoundPeaks(
  seurat_obj  = seurat,
  ref_seqinfo = ref_seqinfo,
  assay_name  = "peaks_integrated"
)

# Now run RegionStats
DefaultAssay(seurat) <- "peaks_integrated"
seurat <- RegionStats(
  object = seurat,
  genome = GRCz11
)

In [ ]:
# data_ids <- c('TDR118''TDR119''TDR124''TDR125''TDR126''TDR127''TDR128')

# # Base directory for the CSV files
# base_dir <- "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/05_SEACells_processed/objects_75cells_per_metacell/"

# # Initialize an empty list to store data frames
# dataframes <- list()

# # Loop through each data ID
# for (data_id in data_ids) {
#   # Construct the file path for the SEACell mapping
#   file_path <- paste0(base_dir, data_id, "_seacells_obs_annotation_ML_coarse.csv")
  
#   # Check if the file exists
#   if (!file.exists(file_path)) {
#     warning(paste("File not found:", file_path))
#     next
#   }
  
#   # Load the SEACell CSV
#   df <- read.csv(file_path)
  
#   # Check if the required columns exist
#   if (!("index" %in% colnames(df)) || !("SEACell" %in% colnames(df))) {
#     stop(paste("Missing required columns in file:", file_path))
#   }
  
#   # Append the dataset suffix to the 'index' column to match Seurat's naming convention
#   df$index <- paste0(df$index, "_", which(data_ids == data_id))  # Appends _1, _2, etc., based on dataset order
  
#   # Append the dataset suffix to the 'SEACell' column for compatibility
#   df$SEACell <- paste0(df$SEACell, "_", which(data_ids == data_id))
  
#   # Add the dataset ID as a separate column for tracking
#   df$data_id <- data_id
  
#   # Append to the list of data frames
#   dataframes[[data_id]] <- df[, c("index", "SEACell", "data_id")]
# }

# # Concatenate all data frames into one
# concatenated_df <- do.call(rbind, dataframes)

# # Save the concatenated mapping to a CSV file
# write.csv(concatenated_df, "concatenated_cell_metacell_mapping.csv", row.names = FALSE)

# # Display the first few rows of the concatenated data frame
# head(concatenated_df)

In [3]:
# # step 1. add the genome annotation
# # path to the GTF file
# gff_path = "/hpc/reference/sequencing_alignment/alignment_references/"
# gref_path = paste0(gff_path, "zebrafish_genome_GRCz11/genes/genes.gtf.gz")
# gtf_zf <- rtracklayer::import(gref_path)

# # make a gene.coord object
# gene.coords.zf <- gtf_zf
# # filter out the entries without the gene_name
# gene.coords.zf <- gene.coords.zf[! is.na(gene.coords.zf$gene_name),]

# # only keep the regions within standard chromosomes
# gene.coords.zf <- keepStandardChromosomes(gene.coords.zf, pruning.mode = 'coarse')
# # name the genome - GRCz11
# genome(gene.coords.zf) <- 'GRCz11'

# # copy the "gene_id" for the "tx_id" and "transcript_id" 
# gene.coords.zf$tx_id <- gene.coords.zf$gene_id
# gene.coords.zf$transcript_id <- gene.coords.zf$gene_id

In [8]:
# DefaultAssay(seurat) <- "peaks_integrated"

# # first compute the GC content for each peak
# seurat <- RegionStats(seurat, genome = BSgenome.Drerio.UCSC.danRer11)

Warning message in RegionStats.default(object = regions, genome = genome, verbose = verbose, :
“Not all seqlevels present in supplied genome”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from peaks_integrated_ to peaksintegrated_”


In [46]:
# extract the current peaks
peak_ranges <- granges(seurat[["peaks_integrated"]])

# We compare the set of peak chroms to what's in the reference
peak_ranges <- dropSeqlevels(
  x = peak_ranges,
  value = setdiff(seqlevels(peak_ranges), seqlevels(GRCz11)),
  pruning.mode = "coarse"
)

# Trim out-of-bounds intervals(peaks)
peak_ranges <- trim(peak_ranges)
# make sure that the peaks have width (non-zero)
peak_ranges <- peak_ranges[width(peak_ranges) > 0]

seurat[["peaks_integrated"]] <- SetAssayData(
  object = seurat[["peaks_integrated"]],
  slot   = "ranges",
  new.data = peak_ranges
)

Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from peaks_integrated_ to peaksintegrated_”


In [50]:
peak_ranges <- granges(seurat[["peaks_integrated"]])
offender <- peak_ranges[
  seqnames(peak_ranges) == "3" &
    start(peak_ranges) <= 62628504 &
    end(peak_ranges) >= 62628283
]
offender

GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]        3 62628283-62628504      *
  -------
  seqinfo: 26 sequences from an unspecified genome; no seqlengths

In [55]:
peak_ranges <- peak_ranges[! (
  seqnames(peak_ranges) == "3" &
  start(peak_ranges) == 62628283 &
  end(peak_ranges)   == 62628504
)]

In [57]:
seurat[["peaks_integrated"]] <- SetAssayData(
  object = seurat[["peaks_integrated"]],
  slot   = "ranges",
  new.data = peak_ranges
)

# first compute the GC content for each peak
DefaultAssay(seurat) <- "peaks_integrated"
seurat <- RegionStats(seurat, genome = GRCz11)

ERROR: Error in SetAssayData.ChromatinAssay(object = seurat[["peaks_integrated"]], : Number of ranges provided is not equal to the number
           of features in the assay


In [58]:
peak_to_remove <- "3-62628283-62628504"
peak_assay <- seurat[["peaks_integrated"]]

# 1) Exclude the peak from the assay features
features_keep <- setdiff(rownames(peak_assay), peak_to_remove)

# 2) Subset the ChromatinAssay using these features
peak_assay_subset <- subset(
  x = peak_assay,
  features = features_keep
)

peak_assay_subset

ChromatinAssay data with 640833 features for 95196 cells
Variable features: 0 
Genome: 
Annotation present: TRUE 
Motifs present: FALSE 
Fragment files: 49 

In [60]:
# Check that names(peak_ranges) match rownames(peak_assay_subset)
all(names(peak_ranges) == rownames(peak_assay_subset))  # should be TRUE

peak_assay_subset <- SetAssayData(
  object  = peak_assay_subset,
  slot    = "ranges",
  new.data = peak_ranges
)

seurat[["peaks_integrated"]] <- peak_assay_subset

[1] TRUE

In [11]:
# standardize the peak naming convention (chr_num-start-end)

library(GenomicRanges)
library(GenomeInfoDb)

# Extract peaks from your ChromatinAssay
peaks_gr <- granges(seurat[["peaks_integrated"]])

# If your peaks look like "1", "2" and you need them to be "chr1", "chr2", ...
# rename each level to add "chr".
old_levels <- seqlevels(peaks_gr)
# old_levels is something like c("1", "2", "3", ...)
new_levels <- paste0("chr", old_levels)
peaks_gr <- renameSeqlevels(peaks_gr, new_levels)

# Now put the updated GRanges back into the assay
seurat[["peaks_integrated"]] <- SetAssayData(
  object = seurat[["peaks_integrated"]],
  slot   = "ranges",
  new.data = peaks_gr
)


In [18]:
# Extract peak ranges
peak_ranges <- granges(seurat[["peaks_integrated"]])

# Check actual seqlengths for chr3 in BSgenome
chr3_len <- seqlengths(BSgenome.Drerio.UCSC.danRer11)["chr3"]
chr3_len

# Subset just the chr3 peaks
chr3_peaks <- peak_ranges[seqnames(peak_ranges) == "chr3"]

# Find peaks that exceed this chromosome boundary
invalid_peaks <- chr3_peaks[end(chr3_peaks) > chr3_len]
invalid_peaks

chr3 
62628489

GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]     chr3 62628283-62628504      *
  -------
  seqinfo: 26 sequences from an unspecified genome; no seqlengths

In [1]:
# this code is now moved to an R script, for slurm submission

In [79]:
RemoveOutOfBoundPeaks <- function(
  seurat_obj,
  ref_seqinfo,
  assay_name = "peaks_integrated"
) {
  library(GenomicRanges)
  library(GenomeInfoDb)

  # 1) Extract the ChromatinAssay and its peaks (GRanges)
  peak_assay <- seurat_obj[[assay_name]]
  peak_ranges <- granges(peak_assay)

  # 2) Get a named vector of chromosome lengths
  #    (from a FaFile or DNAStringSet, e.g. seqinfo(GRCz11))
  chr_lens <- seqlengths(ref_seqinfo)

  # 3) Identify which peaks are truly "in bound":
  #    - On a chromosome that exists in ref_seqinfo
  #    - start >= 1
  #    - end <= chromosome length
  seqn   <- as.character(seqnames(peak_ranges))
  starts <- start(peak_ranges)
  ends   <- end(peak_ranges)

  in_ref <- seqn %in% names(chr_lens)
  in_lower_bound <- (starts >= 1)
  in_upper_bound <- ends <= chr_lens[seqn]  # references the length for each chromosome

  keep_vec <- in_ref & in_lower_bound & in_upper_bound

  # 4) Turn those valid intervals into the "peak names" used by the ChromatinAssay
  # Usually, Signac rownames look like "chr3:62628283-62628504" or "3:62628283-62628504"
  old_feature_names <- rownames(peak_assay)  # existing rownames in the assay
  # The new "kept" features (in-bound)
  features_keep <- old_feature_names[keep_vec]

  # 5) Subset the ChromatinAssay in one go
  #    This automatically updates the counts, meta.features, and ranges
  peak_assay_subset <- subset(
    x        = peak_assay,
    features = features_keep
  )

  # 6) Place it back into the Seurat object
  seurat_obj[[assay_name]] <- peak_assay_subset

  message(
    sum(!keep_vec), " out-of-bound peaks removed; ", 
    sum(keep_vec), " remain."
  )
  return(seurat_obj)
}

In [80]:
# Suppose GRCz11 is a FaFile or DNAStringSet with seqinfo(GRCz11) => valid zebrafish chromosome lengths
ref_seqinfo <- seqinfo(GRCz11)

# Remove out-of-bound peaks
seurat <- RemoveOutOfBoundPeaks(
  seurat_obj  = seurat,
  ref_seqinfo = ref_seqinfo,
  assay_name  = "peaks_integrated"
)

# Now run RegionStats
DefaultAssay(seurat) <- "peaks_integrated"
seurat <- RegionStats(
  object = seurat,
  genome = GRCz11
)

2 out-of-bound peaks removed; 640831 remain.



In [82]:
seurat[["peaks_integrated"]]

ChromatinAssay data with 640831 features for 95196 cells
Variable features: 0 
Genome: 
Annotation present: TRUE 
Motifs present: FALSE 
Fragment files: 49 

In [83]:
# save the intermediate object (without two peaks)
saveRDS(seurat, file="/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/01_Signac_processed/integrated_RNA_ATAC_ga_master.rds")

In [84]:
seurat

An object of class Seurat 
726060 features across 95196 samples within 5 assays 
Active assay: peaks_integrated (640831 features, 0 variable features)
 4 other assays present: RNA, SCT, integrated, Gene.Activity
 9 dimensional reductions calculated: integrated_lsi, umap, integrated_pca, umap.rna, umap.atac, wnn.umap, umap.rna.3D, umap.atac.3D, wnn.umap.3D

### Run RegionStats and compute the peak-gene linkage

In [85]:
DefaultAssay(seurat) <- "peaks_integrated"
seurat <- RegionStats(
  object = seurat,
  genome = GRCz11  # your FaFile or DNAStringSet
)

In [86]:
head(seurat[["peaks_integrated"]]@meta.features)

,count,percentile,AA,AC,AG,AT,CA,CC,CG,CT,⋯,GA,GC,GG,GT,TA,TC,TG,TT,GC.percent,sequence.length
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1-32-526,517,0.2852843,53,29,38,40,44,15,7,24,⋯,25,30,17,29,38,17,39,49,38.78788,495
1-2372-3057,1585,0.7012488,62,35,54,42,47,55,8,53,⋯,47,24,35,40,37,49,48,49,45.04373,686
1-3427-4032,3556,0.8576177,69,29,46,60,51,7,6,19,⋯,39,20,26,45,44,27,52,65,35.14851,606
1-4469-7268,30577,0.9831266,217,210,207,156,255,101,67,207,⋯,161,168,108,166,157,152,221,246,44.07143,2800
1-9541-9969,6469,0.9160886,14,32,26,23,44,21,18,43,⋯,21,34,17,26,16,39,37,17,52.44755,429
1-11007-12962,71779,0.9969602,127,128,121,134,151,84,87,110,⋯,124,119,107,157,108,101,192,105,48.05726,1956


## Check the result of the LinkPeaks

In [ ]:
seurat <- readRDS("/hpc//projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data//01_Signac_processed/peak_gene_links.csv")